# The transfer flow, smoke test

The same stages as `Flow_Transfer.ipynb`, made cheap enough to check the wiring on szenes:
- one target (the G1) with two sources (the Franka and the hand);
- VGG19 only, with a 32-dimensional latent;
- one seed and K ∈ {10, all};
- every 25th frame (20 frames per demonstration);
- one epoch for every encoder and Conv-VAE, 10 meta steps, and 50 regressor steps.

All designs are included; D4a runs on the Conv-VAE only, since the ViT is left out. The numbers mean nothing. The flow checks that every stage runs and that the comparison is written.

In [ ]:
import sys
sys.path.append("../..")
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import pathlib
from flow import setup_flow, run_flow, display_flow_report
from visual_proprioception.transfer.transfer_flow import DESIGNS, FAMILIES, TransferFlow, list_available_packs

In [ ]:
flow_name = "Transfer_smoke_00"
# the datasets of this machine: data/expruns/transfer_datasets/<datasets_run>.yaml
datasets_run = "szenes"
packs = ["panda", "g1", "hand"]
targets = ["g1"]
backbones = ["vgg19"]
designs = DESIGNS
latent_size = 32
seeds = [0]
k_shots = [10, None]
frame_step = 25
epochs_sp = 1
epochs_vae = 1
meta_steps = 10
regressor_steps = 50
regressors = ["mlp", "ridge"]
eval_group = "vp_testing"
creation_style = "exist-ok"

In [ ]:
# the packs available on this machine (see DESIGN-Transfer.md, "Datasets on other machines")
list_available_packs()

expruns_path, results_path, notebooks_path = setup_flow(flow_name, FAMILIES)
builder = TransferFlow({
    "flow_name": flow_name, "datasets_run": datasets_run, "packs": packs, "targets": targets,
    "backbones": backbones, "designs": designs, "latent_size": latent_size, "seeds": seeds,
    "k_shots": k_shots, "frame_step": frame_step, "epochs_sp": epochs_sp, "epochs_vae": epochs_vae,
    "meta_steps": meta_steps, "regressor_steps": regressor_steps, "regressors": regressors,
    "eval_group": eval_group, "creation_style": creation_style})
builder.import_packs()
expruns, compare_run = builder.build()
print(f"{len(expruns)} stages, of which {len(builder.vp_runs)} regressor stages")

In [ ]:
flow_error = None
try:
    run_flow(expruns, expruns_path, results_path, notebooks_path)
except Exception as error:
    flow_error = error

In [ ]:
final_results_path = pathlib.Path(results_path, "visual_proprioception_transfer_compare", compare_run)
report = display_flow_report(
    expruns, results_path, notebooks_path, final_results_path,
    flow_error=flow_error,
    preview_files=sorted(final_results_path.glob("comparison_*.png")))
if flow_error is not None:
    raise flow_error
if not report["all-results-present"]:
    raise Exception("Flow finished without all expected stage results.")